# Classification Experiments

In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path(os.getcwd())

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"

sys.path.append(str(ROOT))

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [8]:
def evaluate_predictions(y_true, y_pred, name):
    print(f"\n=== {name} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="macro", zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, average="macro", zero_division=0))
    print("F1       :", f1_score(y_true, y_pred, average="macro", zero_division=0))
    print("MCC      :", matthews_corrcoef(y_true, y_pred))

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

### Load data

In [4]:
train_data = pd.read_pickle(DATA/'ml_datasets2/L2_TRAIN.pkl')

In [5]:
features = train_data.iloc[:, :10]
target = train_data[['source']]

**SPLIT**

In [6]:
X = features.copy()
Y = target.copy()

In [7]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=0, stratify=Y['source'])

In [22]:
print(X_train.shape)
print(X_val.shape)

### Initialize  baseline RF 

In [23]:
base_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

### 1. Environment

In [24]:
Y_train_1 = Y_train['source']
Y_val_1 = Y_val['source']

le_1 = LabelEncoder()
Y_train_source = le_1.fit_transform(Y_train_1)
Y_val_source = le_1.transform(Y_val_1)

In [25]:
base_rf.fit(X_train, Y_train_source)

In [27]:
Y_pred_val = base_rf.predict(X_val)

env_order = ['glc', 'ac', 'lcts', 'glyc', 'succ', 'arg_glc', 'arg_ac', 'arg_glyc',
             'nh4_glc', 'nh4_ac', 'no3_glc', 'no3_glyc']

cm = confusion_matrix(le_1.inverse_transform(Y_val_source), 
                      le_1.inverse_transform(Y_pred_val),
                      labels=env_order)

disp = ConfusionMatrixDisplay(cm, display_labels=env_order)

disp.plot(cmap='Blues', xticks_rotation=90)
plt.title("All Source Lables")
plt.show()

In [28]:
evaluate_predictions(
    Y_val_source,
    Y_pred_val,
    "RF pipeline: Stage 1 source_type"
)

In [29]:
print(classification_report(
    Y_val_source,
    Y_pred_val,
    target_names=le_1.classes_
))

### 2. Sampled nutrient

In [30]:
Y_train_2 = Y_train['source'].str.split('_').str[0]
Y_val_2 = Y_val['source'].str.split('_').str[0]

le_2 = LabelEncoder()

Y_train_source2 = le_2.fit_transform(Y_train_2)
Y_val_source2 = le_2.transform(Y_val_2)

In [31]:
base_rf.fit(X_train, Y_train_source2)

In [32]:
Y_pred_val2 = base_rf.predict(X_val)

env_order = ['glc', 'ac', 'lcts', 'glyc', 'succ', 'arg', 'nh4', 'no3']

cm = confusion_matrix(le_2.inverse_transform(Y_val_source2), 
                      le_2.inverse_transform(Y_pred_val2),
                      labels=env_order)

disp = ConfusionMatrixDisplay(cm, display_labels=env_order)

disp.plot(cmap='Blues', xticks_rotation=90)
plt.title("Limiting nutrient")
plt.show()

In [33]:
evaluate_predictions(
    Y_val_source2,
    Y_pred_val2,
    "RF pipeline: Stage 1 source_type"
)

In [34]:
print(classification_report(
    Y_val_source2,
    Y_pred_val2,
    target_names=le_2.classes_
))

### 3. Binary

In [14]:
Y_train_3 = Y_train['source'].str.contains(r'arg|nh4|no3', case=False, na=False).map({True: 'Nitrogen', False: 'Carbon'})
Y_val_3 = Y_val['source'].str.contains(r'arg|nh4|no3', case=False, na=False).map({True: 'Nitrogen', False: 'Carbon'})

le3 = LabelEncoder()
Y_train_bin = le3.fit_transform(Y_train_3)
Y_val_bin = le3.transform(Y_val_3)

In [39]:
base_rf.fit(X_train, Y_train_bin)

In [19]:
Y_pred_val_3 = base_rf.predict(X_val)

cm = confusion_matrix(Y_val_bin, Y_pred_val_3)
disp = ConfusionMatrixDisplay(cm, display_labels=le3.classes_)

disp.plot(cmap='Blues')
plt.title("Carbon vs. Nitrogen")
plt.show()

In [20]:
evaluate_predictions(
    Y_val_bin,
    Y_pred_val_3,
    "RF pipeline: Stage 1 source_type"
)

In [131]:
accuracy = accuracy_score(Y_val_bin, Y_pred_val_3)
precision = precision_score(Y_val_bin, Y_pred_val_3)
recall = recall_score(Y_val_bin, Y_pred_val_3)
f1 = f1_score(Y_val_bin, Y_pred_val_3)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")

### 4. Isolated Carbon and Nitrogen 

In [15]:
X_train_c = X_train[Y_train_3 == 'Carbon']
X_val_c = X_val[Y_val_3 == 'Carbon']

Y_train_c = Y_train[Y_train_3 == 'Carbon']
Y_val_c = Y_val[Y_val_3 == 'Carbon']

#######################################################
X_train_n = X_train[Y_train_3 == 'Nitrogen']
X_val_n = X_val[Y_val_3 == 'Nitrogen']

Y_train_n = Y_train[Y_train_3 == 'Nitrogen']
Y_val_n = Y_val[Y_val_3 == 'Nitrogen']

In [16]:
Y_train_c = Y_train_c['source']
Y_val_c = Y_val_c['source']

le_c = LabelEncoder()

Y_train_enc_c = le_c.fit_transform(Y_train_c)
Y_val_enc_c = le_c.fit_transform(Y_val_c)

#########################################

le_n = LabelEncoder()

Y_train_n = Y_train_n['source']
Y_val_n = Y_val_n['source']
# Y_train_n = Y_train_n['source'].str.split('_').str[0]
# Y_val_n = Y_val_n['source'].str.split('_').str[0]

Y_train_enc_n = le_n.fit_transform(Y_train_n)
Y_val_enc_n = le_n.fit_transform(Y_val_n)

**CARBON**

In [54]:
base_rf.fit(X_train_c, Y_train_enc_c)

In [24]:
Y_pred_val_c = base_rf.predict(X_val_c)

env_order = ['glc', 'ac', 'lcts', 'glyc', 'succ']

cm = confusion_matrix(le_c.inverse_transform(Y_val_enc_c), 
                      le_c.inverse_transform(Y_pred_val_c),
                      labels=env_order)

disp = ConfusionMatrixDisplay(cm, display_labels=env_order)

disp.plot(cmap='Blues')
plt.title("Carbon sources")
plt.show()

In [27]:
evaluate_predictions(
    Y_val_c,
    le_c.inverse_transform(Y_pred_val_c),
    "RF pipeline: Stage 1 source_type"
)

In [137]:
print(classification_report(
    Y_val_enc_c,
    Y_pred_val_c,
    target_names=le_c.classes_
))

**NITROGEN**

In [17]:
base_rf.fit(X_train_n, Y_train_enc_n)

In [19]:
Y_pred_val_n = base_rf.predict(X_val_n)

env_order = ['arg_glc', 'arg_ac', 'arg_glyc',
             'nh4_glc', 'nh4_ac', 'no3_glc', 'no3_ac', 'no3_glyc']

cm = confusion_matrix(le_n.inverse_transform(Y_val_enc_n), 
                      le_n.inverse_transform(Y_pred_val_n),
                      labels=env_order)

disp = ConfusionMatrixDisplay(cm, display_labels=env_order)

disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Nitrogen sources")
plt.show()

In [153]:
print(classification_report(
    Y_val_enc_n,
    Y_pred_val_n,
    target_names=le_n.classes_
))

In [20]:
evaluate_predictions(
    Y_val_n,
    le_n.inverse_transform(Y_pred_val_n),
    "RF pipeline: Stage 1 source_type"
)

In [29]:
Y_pred_val_n = base_rf.predict(X_val_n)

env_order = ['arg', 'nh4', 'no3']

cm = confusion_matrix(le_n.inverse_transform(Y_val_enc_n), 
                      le_n.inverse_transform(Y_pred_val_n),
                      labels=env_order)

disp = ConfusionMatrixDisplay(cm, display_labels=env_order)

disp.plot(cmap='Blues')
plt.title("Nitrogen sources")
plt.show()

In [30]:
evaluate_predictions(
    Y_val_n,
    le_n.inverse_transform(Y_pred_val_n),
    "RF pipeline: Stage 1 source_type"
)

In [ ]:
print(classification_report(
    Y_val_enc_n,
    Y_pred_val_n,
    target_names=le_n.classes_
))